In [ ]:
import xgboost as xgb 
import pandas as pd 
import sklearn as sk 
import re 
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold 
from sklearn.model_selection import GridSearchCV 
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import accuracy_score 
from scipy.linalg import sqrtm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
%matplotlib inline

In [ ]:
def read_data(file,label):
    data = pd.read_csv(f'{file}')
    data['label'] = label
    return data

In [ ]:
a1MeV = read_data('train_data/select_data/m_a = 1   MeV.csv',1)
a10MeV = read_data('train_data/select_data/m_a = 10  MeV.csv',1)
a100MeV = read_data('train_data/select_data/m_a = 100 MeV.csv',1)
a1GeV = read_data('train_data/select_data/m_a = 1   GeV.csv',1)
a10GeV = read_data('train_data/select_data/m_a = 10  GeV.csv',1)
aME = read_data('train_data/select_data/Mono-gamma.csv',0)

In [ ]:
all_data = pd.concat([a1MeV,a10MeV,a100MeV,a1GeV,a10GeV,aME],axis = 0)
all_data = all_data.sample(frac=1).reset_index(drop=True)

In [ ]:
a1GeV.columns,all_data.shape

In [ ]:
X = all_data[['MissingET_MET_0', 'photon_energy', 'photon_theta_cos']]
y = all_data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


# 创建模型

In [ ]:
clf = xgb.XGBClassifier(learning_rate=0.02, 
                        max_depth=4, 
                        objective='binary:logistic',
                        eval_metric=['logloss', 'auc', 'error'],
                        tree_method='hist',  # 使用 CPU 的直方法
                        device='cuda',       # 指定使用 GPU
                        verbosity=2)

In [ ]:
model = xgb.XGBClassifier(learning_rate = 0.02 , max_depth=6, n_estimators=300,
                        objective='binary:logistic',
                        eval_metric=['logloss','auc','error'])

In [ ]:
model.fit(X_train,y_train)

In [ ]:
model.save_model('clf_ALP_6_300')